In [4]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from PIL import Image
import torch

# -----------------------------
# Model setup
# -----------------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
device = "cuda"

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device)
model.eval()

# -----------------------------
# Load image
# -----------------------------
image_path = r"C:\smart_invoice\invoices\WhatsApp Image 2026-02-28 at 4.28.18 PM.jpeg"
image = Image.open(image_path).convert("RGB")

# -----------------------------
# Prompt
# -----------------------------
prompt = """
You are an AI invoice extractor. Extract the following fields from the invoice image attached in the message:

{
  "supplier_name": "Name of the supplier",
  "total_price": 123.45,
  "payment_method": "Cash, Card, or Other",
  "date_time": "YYYY-MM-DD HH:MM",
  "confidence": 0.95
}

Rules:
- Return JSON only, no explanations.
- If a field cannot be found, use empty string "" or null.
- Fill the confidence field with a float (0-1) indicating your confidence in the extraction.
"""

# -----------------------------
# Messages (image + text)
# -----------------------------
messages = [
    {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}
]

# -----------------------------
# Apply chat template WITHOUT tokenize
# -----------------------------
chat_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# -----------------------------
# Prepare model inputs
# -----------------------------
inputs = processor(
    text=chat_prompt,
    images=[image],
    return_tensors="pt"
).to(device)

# -----------------------------
# Generate
# -----------------------------
outputs = model.generate(**inputs, max_new_tokens=512)
raw_text = processor.decode(outputs[0], skip_special_tokens=True)

print(raw_text)

Fetching 2 files: 100%|██████████| 2/2 [00:00<?, ?it/s]


OSError: The paging file is too small for this operation to complete. (os error 1455)